# Generator parameter sweep

Phase 1 of improving the generator in isolation: expose the decoding surface on
`google/gemma-3-1b-it`, then move one setting at a time.

Everything else is held fixed. The snapshot is untouched, the same 32 verified
golden rows are scored every run, the prompt is built through the serving path,
and the judge configuration comes from `configs/base_config.yaml` unchanged. One
configuration per run, one MLflow run per repeat, and the sweep itself is the
record.

The parameter surface, meaning everything that can potentially be changed:

| Group | Parameters | What it would move |
| --- | --- | --- |
| Sampling shape | `do_sample`, `temperature`, `top_p`, `top_k`, `min_p`, `typical_p`, `epsilon_cutoff`, `eta_cutoff` | Whether the model samples at all, and from how wide a slice of the distribution. Turning `do_sample` on is the largest single behavioural change available. |
| Degeneration | `repetition_penalty`, `no_repeat_ngram_size` | Looping and padding, the two failure modes a small model at greedy decoding falls into. The only group that can help without sampling. |
| Length | `max_new_tokens`, `min_new_tokens` | How long the answer may be, and how short it may not be. `min_new_tokens` is the blunt way to stop an answer closing before it has named the dish. |
| Stopping | `stop_strings` | Cuts generation at a literal string. Needs the tokenizer handed to `generate` alongside it. |
| Reproducibility | `torch.manual_seed` | Not a `generate` argument in this stack, so a fixed seed means seeding torch before the call. |
| Not a knob | `max_input_tokens`, `truncation`, `return_tensors` | Carried by the serving generator's config and never read. Listed so they are not mistaken for levers. |

Rules this notebook follows:

- One axis at a time. `SETTINGS` in the knobs cell is the only thing that changes between runs.
- While `do_sample` is False a run is deterministic, so one pass is the whole answer and a repeat would only cost judge calls. Once sampling is on, a configuration runs three times and is reported as a mean and a range, because the service sets no seed either and what we record should be what a user gets.
- Every run logs the full decoding set, the model id, the snapshot, the dataset key, the row count, the repeat index, and the judge configuration to MLflow under `rag-generator-tuning`.
- The generated text is never summarised away. The samples cells print the actual answers, because the check failing here is a matter of how an answer opens.

## Setup

Run from this directory. The cells above the fold are the only ones that need
changing between runs.

In [ ]:
import json
import sys
import tempfile
import time
from pathlib import Path

import mlflow
import torch
from transformers import AutoModelForCausalLM

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "src" / "rag").is_dir() and REPO_ROOT.parent != REPO_ROOT:
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from scripts._eval_common import (
    begin_mlflow_run,
    build_judge,
    build_prompt_builder,
    judge_config,
    load_recipes_by_id,
    load_rows,
    load_snapshot,
    resolve_recipes_path,
    resolve_snapshot_folder,
    tokenizer_for,
)
from src.rag.core.config import DATA_PATHS
from src.rag.data.recipe_format import recipe_to_string
from src.rag.evaluation.generator_evaluator import GeneratorEvaluator
from src.rag.evaluation.judge_prompts import judge_prompt_hash

print("repo root:", REPO_ROOT)

## The knobs

`SETTINGS` is the whole decoding surface. A value of `None` means the key is not
passed to `generate` at all, so the library default applies and a run differs
from the baseline only in the settings given a value.

As written this reproduces the serving generator exactly: greedy, 2000 new
tokens, no sampler.

In [ ]:
SNAPSHOT_TARGET = "latest"
EXPERIMENT = "rag-generator-tuning"
RUN_PREFIX = "gen-tuning"

# The decoding surface. Only keys with a value reach generate().
SETTINGS = {
    "model_id": "google/gemma-3-1b-it",
    "max_new_tokens": 2000,
    "min_new_tokens": None,
    "do_sample": False,
    "temperature": None,
    "top_p": None,
    "top_k": None,
    "min_p": None,
    "typical_p": None,
    "repetition_penalty": None,
    "no_repeat_ngram_size": None,
    "stop_strings": None,
}

# This stack has no seed keyword, so reproducibility is a manual torch seed.
# Left unset, which is what the service does.
SEED = None

# A deterministic configuration needs one pass; a sampled one needs a spread.
GREEDY_REPEATS = 1
SAMPLED_REPEATS = 3


def repeats_for(settings):
    """How many times a configuration is run: one pass if it is deterministic."""
    return SAMPLED_REPEATS if settings["do_sample"] else GREEDY_REPEATS

## Load the snapshot and the scoring path

Loaded once and reused. Every run in this notebook differs only in the decoder,
so the snapshot, the rows, the corpus, the tokenizer, the prompt builder, and
the judge configuration are all part of the fixed side of the comparison.

In [ ]:
def load_eval_context():
    """The snapshot, golden rows, corpus, prompt path, and judge configuration."""
    folder = resolve_snapshot_folder(SNAPSHOT_TARGET)
    snapshot = load_snapshot(folder)
    tokenizer = tokenizer_for(snapshot)
    return {
        "folder": folder,
        "snapshot": snapshot,
        "recipes_by_id": load_recipes_by_id(resolve_recipes_path(snapshot)),
        "rows": load_rows(DATA_PATHS["eval_answers"], limit=None, verified=True),
        "tokenizer": tokenizer,
        "prompt_builder": build_prompt_builder(snapshot, tokenizer),
        "judge_cfg": judge_config(),
    }


CTX = load_eval_context()
print("snapshot:", CTX["folder"])
print("rows:", len(CTX["rows"]), "| query types:", sorted({r["query_type"] for r in CTX["rows"]}))
print("judge:", CTX["judge_cfg"]["model_id"], "| calls:", CTX["judge_cfg"]["judge_calls"])

## A generator with the decoding surface exposed

The serving `Generator` hardcodes `do_sample=False` and passes nothing else, so
this class exists to make those settings values. It renders the prompt
identically: same tokenizer, same chat template call, same message list. Only
the call into `generate` differs, which is what makes a comparison between two
configurations a comparison of decoding rather than of prompting.

The weights load on first use, so building this object is cheap and a
configuration that is never run costs nothing.

In [ ]:
class SamplingGenerator:
    """A generator whose decoding settings are values rather than constants."""

    def __init__(self, settings, tokenizer):
        self.settings = settings
        self.tokenizer = tokenizer
        self.model = None

    def load(self):
        """Load the weights on first use."""
        if self.model is None:
            self.model = AutoModelForCausalLM.from_pretrained(
                self.settings["model_id"],
                torch_dtype=torch.bfloat16,
                device_map="auto",
            )
        return self.model

    def decoding_kwargs(self):
        """The settings that carry a value, so nothing unset is passed on."""
        return {
            key: value
            for key, value in self.settings.items()
            if key != "model_id" and value is not None
        }

    def generate(self, system, user):
        """Generate a response from a (system, user) prompt pair."""
        messages = []
        if system:
            messages.append({"role": "system", "content": system})
        messages.append({"role": "user", "content": user})

        text = self.tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        model_inputs = self.tokenizer([text], return_tensors="pt").to(self.load().device)

        kwargs = self.decoding_kwargs()
        if isinstance(kwargs.get("stop_strings"), str):
            # stop_strings is matched against decoded text rather than token
            # ids, so generate() needs the tokenizer in order to apply it.
            kwargs["tokenizer"] = self.tokenizer

        if SEED is not None:
            torch.manual_seed(SEED)

        with torch.inference_mode():
            generated_ids = self.model.generate(**model_inputs, **kwargs)

        output_ids = generated_ids[0][len(model_inputs["input_ids"][0]):].tolist()
        return self.tokenizer.decode(output_ids, skip_special_tokens=True)


def build_generator(settings):
    return SamplingGenerator(settings, CTX["tokenizer"])


GEN = build_generator(SETTINGS)
print("decoding kwargs:", GEN.decoding_kwargs())
print("repeats per configuration:", repeats_for(SETTINGS))

Before spending a GPU pass, it is worth confirming that the prompt going in is
the serving prompt. This renders the first row's prompt the way the generator
will render it.

In [ ]:
first = CTX["rows"][0]
system, user = CTX["prompt_builder"].build_prompt_from_recipes(
    query=first["query"],
    recipes=[CTX["recipes_by_id"][rid] for rid in first["recipe_ids"]],
    recipe_to_string=recipe_to_string,
)
rendered = CTX["tokenizer"].apply_chat_template(
    [{"role": "system", "content": system}, {"role": "user", "content": user}],
    tokenize=False,
    add_generation_prompt=True,
)
print("chars:", len(rendered), "| tokens:", len(CTX["tokenizer"](rendered)["input_ids"]))
print("-" * 78)
print(rendered[:1500])

## Running one configuration

Each repeat gets its own MLflow run, carrying the settings, the identities, and
the per-row results as an artifact. The returned structure is kept in memory so
the cells below can compare configurations without opening the store.

In [ ]:
def log_params(settings, repeats, index, label):
    """Everything a run needs in order to be readable a week later."""
    params = {
        "label": label,
        "generator_model_id": settings["model_id"],
        "snapshot_name": CTX["folder"],
        "dataset_key": CTX["snapshot"].dataset_key,
        "strategy": CTX["snapshot"].retrieval_strategy,
        "rows": len(CTX["rows"]),
        "repeat_index": index,
        "repeats": repeats,
        "seed": "none" if SEED is None else SEED,
        "judge_model_id": CTX["judge_cfg"]["model_id"],
        "judge_temperature": CTX["judge_cfg"]["temperature"],
        "judge_calls": CTX["judge_cfg"]["judge_calls"],
        "judge_prompt_hash": judge_prompt_hash(),
    }
    for key, value in settings.items():
        params[f"decode_{key}"] = "none" if value is None else value
    return params


def run_configuration(settings, label, repeats=None):
    """Score one configuration, logging every repeat as its own MLflow run."""
    repeats = repeats or repeats_for(settings)
    generator = build_generator(settings)
    per_repeat = []

    for index in range(1, repeats + 1):
        name = f"{RUN_PREFIX}-{label}-r{index}-{time.strftime('%H%M%S')}"
        evaluator = GeneratorEvaluator(
            prompt_builder=CTX["prompt_builder"],
            generator=generator,
            judge=build_judge(CTX["judge_cfg"]),
            recipes_by_id=CTX["recipes_by_id"],
            judge_calls=CTX["judge_cfg"]["judge_calls"],
        )
        results = evaluator.evaluate(CTX["rows"])

        with begin_mlflow_run(EXPERIMENT, name):
            mlflow.log_params(log_params(settings, repeats, index, label))
            for metric, value in results["slice"].items():
                mlflow.log_metric(metric, value)
            for query_type, scores in results["query_type"].items():
                for metric, value in scores.items():
                    mlflow.log_metric(f"{query_type}.{metric}", value)
            with tempfile.TemporaryDirectory() as tmp:
                out = Path(tmp) / "per_row.json"
                out.write_text(json.dumps(results["per_row"], indent=2))
                mlflow.log_artifact(str(out))

        per_repeat.append(results)
        print(f"repeat {index}/{repeats} logged as {name}")

    return per_repeat


def summarise(per_repeat):
    """Mean, minimum, and maximum of each slice metric across the repeats."""
    summary = {}
    for metric in sorted(per_repeat[0]["slice"]):
        values = [run["slice"][metric] for run in per_repeat]
        summary[metric] = {
            "mean": sum(values) / len(values),
            "min": min(values),
            "max": max(values),
        }
    return summary

## Baseline: the same behaviour through the new class

The first job is not to change anything. It is to run the greedy path through
the new class and confirm the numbers land where the old class put them, which
is what makes every later configuration comparable.

Two things to expect. This is the first golden-context generation run on record,
so there is no like-for-like number to match yet: the only generation figures in
the store come from the end-to-end eval, whose context is retrieval output rather
than the golden recipes. And on a 6 GB card the served model holds most of the
VRAM, so stop the service before running this or the weight load will fail.

Expect roughly two minutes per pass plus the judge calls, which are the slow
part: 32 rows at two calls each.

In [ ]:
BASELINE = run_configuration(SETTINGS, label="baseline")

for metric, stats in summarise(BASELINE).items():
    print(f"  {metric}: mean={stats['mean']:.4f} min={stats['min']:.4f} max={stats['max']:.4f}")

In [ ]:
# The only generation numbers on record come from the end-to-end run on
# 2026-09-12, whose context is retrieval output at serving_k 3 rather than the
# golden recipes. It is a reference point, not a like-for-like baseline.
E2E_REFERENCE = {
    "structure_pass_rate": 0.0625,
    "mean.constraint_satisfaction": 0.71875,
    "mean.query_relevance": 1.484375,
    "mean.faithfulness": 0.71875,
    "mean.expectedness": 0.96875,
}

print(f"{'metric':<34}{'golden':>10}{'e2e':>10}")
for metric, value in E2E_REFERENCE.items():
    golden = BASELINE[0]["slice"][metric]
    print(f"{metric:<34}{golden:>10.4f}{value:>10.4f}")

## Reading the answers

The structural score passes a row only when all six checks pass, and five of
them were already near-perfect. So the interesting cells are the ones showing
which check fails and how the answer opens.

In [ ]:
def show_samples(results, count=3, chars=700):
    """Print the opening of a few answers, since the failing check is an opener."""
    for row in results["per_row"][:count]:
        failed = [flag for flag, ok in row["structural"].items() if not ok]
        print("=" * 78)
        print(f"[{row['query_type']}] {row['query']}")
        print(f"failed checks: {failed or 'none'}")
        print("-" * 78)
        print(row["generated"][:chars].strip())
        print()


show_samples(BASELINE[0])

In [ ]:
FLAG = "has_name"

failing = [row for row in BASELINE[0]["per_row"] if not row["structural"][FLAG]]
print(f"{FLAG}: {len(BASELINE[0]['per_row']) - len(failing)} of {len(BASELINE[0]['per_row'])} rows pass")
for row in failing:
    print(f"  [{row['query_type']}] {row['generated'][:95].strip()!r}")

In [ ]:
for query_type, scores in BASELINE[0]["query_type"].items():
    flags = " ".join(
        f"{flag}={value:.2f}" for flag, value in sorted(scores.items()) if flag.startswith("flag.")
    )
    print(f"[{query_type}]")
    print(f"  structure={scores['structure_pass_rate']:.4f}")
    print(f"  {flags}")

## Sweeping

Copy the knobs cell's `SETTINGS` into a new object, change one value, and run it.
`run_configuration` returns a list of repeats, one per pass, so two
configurations compare like this:

```python
SLICE_TOP_P_FIXED = run_configuration(settings_a, label="temp-0.7")
summarise(SLICE_TOP_P_FIXED)
```

The rules that keep the comparison honest:

- One value per run. Two moved at once and the result belongs to neither.
- Compare against `BASELINE` from this session, not against the historical
  numbers, since the sampler and the judge both drift with time.
- Judge disagreement runs 12 to 31 percent of rows depending on the dimension, so
  a difference of one row is inside the noise and a difference of four is not.
- Every claim needs the answer text behind it. The samples cells are the check on
  the metrics, not decoration.

## Where the levers sit

These were parked rather than decided, and the baseline above is what decides
what to pick up first:

- Degeneration penalties, the only group that can help while sampling is off.
- Length and stopping, aimed at the opening paragraph. This one moves the gate
  metric by suppressing the symptom rather than by making the model comply, and
  it should be labelled that way if it works.
- Sampling, once there is a reason for it: `do_sample=True` with a temperature
  and a truncation rule, which is where `repeats_for` starts asking for three
  passes.
- The persona and prompt text, which is not a decoding parameter at all and is
  the most likely real cause of the opener. Out of scope here on purpose.

## One more check worth running once

The judge is called twice per row and its two verdicts disagree often enough to
matter. This probe asks whether the provider honours temperature at all, which
decides whether that disagreement is judge noise or provider routing.

In [ ]:
from scripts._eval_common import probe_judge

probe_judge(CTX["judge_cfg"], CTX["rows"][0], CTX["recipes_by_id"])